# Urban Heat Alert — NaaVRE Provenance Demo

This self-contained workflow turns synthetic hourly city temperatures into a heat-alert report. It is designed to make three provenance features visible in one short story.

1. Containerize the three workflow cells below and connect them in order.
2. Save the workflow as `urban-heat-provenance-demo.naavrewf`.
3. Run a baseline with `param_threshold_c = 31.0` and `param_min_hot_hours = 3`.
4. Run a heatwave scenario with `param_threshold_c = 29.5` and `param_min_hot_hours = 2`.
5. Compare the two runs in the Experiment Manager.
6. After the builds, change the detector's `severity = "warning"` line to `severity = "critical"` without rebuilding.
7. On the detector node, use **Jump to Cell** to see the current editable source, then **View Build Source** to see the immutable source used for the build.

> The next cell supplies defaults for local **Run All** execution. Containerize only the three named workflow cells.

In [ ]:
param_seed = 17
param_hours = 24
param_threshold_c = 31.0
param_min_hot_hours = 3
param_scenario_name = "baseline"

## 1. Generate city readings

Creates a small, deterministic dataset with no download and no third-party dependencies.

In [ ]:
# Generate city readings
# ---
# NaaVRE:
#   cell:
#     outputs:
#       - city_readings:
#           type: List
#     params:
#       - param_seed:
#           type: Integer
#           default_value: 17
#       - param_hours:
#           type: Integer
#           default_value: 24
# ...
import math
import random

rng = random.Random(param_seed)
districts = ["Centrum", "West", "Nieuw-West", "Oost"]
city_readings = []
for hour in range(param_hours):
    daily_curve = 6.5 * math.sin((hour - 7) * math.pi / 12)
    for district_index, district in enumerate(districts):
        urban_heat = district_index * 0.7
        temperature_c = 26.0 + daily_curve + urban_heat + rng.uniform(-0.6, 0.6)
        city_readings.append({
            "district": district,
            "hour": hour,
            "temperature_c": round(temperature_c, 1),
        })

print(f"Generated {len(city_readings)} readings across {len(districts)} districts")

## 2. Detect heat hotspots

This is the centerpiece of the demo: its two parameters differ between runs, and its `severity` line is edited after the build to contrast live and historical source.

In [ ]:
# Detect heat hotspots
# ---
# NaaVRE:
#   cell:
#     inputs:
#       - city_readings:
#           type: List
#     outputs:
#       - hotspot_summary:
#           type: List
#     params:
#       - param_threshold_c:
#           type: Float
#           default_value: 31.0
#       - param_min_hot_hours:
#           type: Integer
#           default_value: 3
# ...
from collections import defaultdict

severity = "warning"
hot_hours = defaultdict(list)
for reading in city_readings:
    if reading["temperature_c"] >= param_threshold_c:
        hot_hours[reading["district"]].append(reading)

hotspot_summary = []
for district, readings in sorted(hot_hours.items()):
    if len(readings) >= param_min_hot_hours:
        hotspot_summary.append({
            "district": district,
            "severity": severity,
            "hot_hours": len(readings),
            "peak_temperature_c": max(item["temperature_c"] for item in readings),
        })

print(f"Detected {len(hotspot_summary)} hotspots at or above {param_threshold_c} C")

## 3. Summarize heat alerts

Turns the structured detector output into an immediately readable scenario report.

In [ ]:
# Summarize heat alerts
# ---
# NaaVRE:
#   cell:
#     inputs:
#       - hotspot_summary:
#           type: List
#     outputs:
#       - alert_report:
#           type: String
#     params:
#       - param_scenario_name:
#           type: String
#           default_value: "baseline"
# ...
if hotspot_summary:
    lines = [
        f"- {item['district']}: {item['hot_hours']} hot hours, "
        f"peak {item['peak_temperature_c']} C ({item['severity']})"
        for item in hotspot_summary
    ]
else:
    lines = ["- No districts crossed the alert threshold"]

alert_report = "Scenario: " + param_scenario_name + "\n" + "\n".join(lines)
print(alert_report)

---

### Presenter reminder

Build first. Then edit only `severity = "warning"` to `severity = "critical"` and do **not** rebuild. This preserves the clean live-source versus build-source reveal.